In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

#---------
#1. Read Broze table and store in Dataframe
#-------

df_customer = spark.table("workspace.bronze.customers")
df_products = spark.table("workspace.bronze.products")
df_orders = spark.table("workspace.bronze.orders")


#---------
#2. create new product dataframe

# -------)
df_product_clean = (
    df_products
    .dropDuplicates(["product_id"])
    .filter(F.col("product_id").isNotNull())
    .filter(F.col("unit_price") > 0 )
    .withColumn("category", F.coalesce(F.col("category"), F.lit("Uncategorized")))
  
 
)
#---------
#3. create new customer dataframe

# -------)
df_customer_clean = (
    df_customer
    .dropDuplicates(["customer_id"])
    .filter(F.col("customer_id").isNotNull())
    .filter(F.col("first_name").isNotNull())
    .withColumn("region", F.when(F.col("region").isNotNull(), F.col("region")).otherwise(F.lit("Unknown" )))
    .withColumn("customer_segment", F.coalesce(F.col("customer_segment"), F.lit("Regular"))) 
    .withColumn("signup_date", F.to_date(F.col("signup_date"), "yyyy-MM-dd"))

)

#df_customer_clean.display()
#---------
#4. create new orders dataframe

# -------)
window_spec = Window.partitionBy("order_id").orderBy(F.col("order_date").desc())
df_orders_clean = (
    df_orders
    .withColumn("row_num", F.row_number().over(window_spec))
    .filter(F.col("row_num") == 1)
    .drop("row_num")
    .filter(F.col("order_id").isNotNull())
    .filter(F.col("order_date").isNotNull())
    .filter(F.col("order_status").isNotNull())
    .withColumn("product_id", F.coalesce(F.col("product_id"), F.lit(-1)))
    .filter(F.col("product_id").isNotNull())
    .filter(F.col("quantity") > 0)
    
)
 

print("After cleaning -> customers:", df_customer_clean.count(),
      "| products:", df_product_clean.count(),
      "| orders:", df_orders_clean.count())
 
     

In [0]:
#------------------
# 5. Create enriched orders dataframe
#------------------

df_silver = (
df_orders_clean.alias("o")
.join(df_customer_clean.alias("c"), "customer_id", "left")
.join(df_product_clean.alias("p"), "product_id", "left")
.select(
    F.col("o.order_id"),
    F.col("o.order_date"),
    F.col("o.order_status"),
    F.col("o.payment_method"),
    F.col("o.quantity"),
    F.col("o.customer_id"),
    F.col("c.region"),
    F.col("c.customer_segment"),
    F.col("o.product_id"),
    F.col("p.category"),
    F.col("p.unit_price"),
    F.col("p.cost_price")
)
#Dervied column or transformation
.withColumn("gross_amount", F.round(F.col("quantity") * F.col("unit_price"),2))
.withColumn("cost_amount", F.round(F.col("quantity") * F.col("cost_price"),2))
.withColumn("profit_amount", F.round(F.col("gross_amount") - F.col("cost_amount"),2))
.withColumn("is_revenue_recognized", F.when(F.upper(F.col("order_status")) == "COMPLETED", F.lit(True)).otherwise(F.lit(False)))
.withColumn("Order_year",F.year("order_date"))
.withColumn("Order_month",F.month("order_date"))

)
#display(df_silver.limit(10))
#display(df_silver.filter(F.col("region").isNull()))
display(df_silver.filter(F.col("category").isNull()))
 

In [0]:
#-------
#6.Data Quality Check
#---------

null_customers = df_silver.filter(F.col("region").isNull()).count()
null_products = df_silver.filter(F.col("category").isNull()).count()

print(f"Orders with missing customer match: {null_customers}")
print(f"Orders with missing product match: {null_products}")

# Fail loudly if referential integrity is badly broken (adjust threshold as needed)
assert null_customers < df_silver.count() * 0.01, "Too many orphaned customer_id references!"
assert null_products < df_silver.count() * 0.01, "Too many orphaned product_id references!"

In [0]:
from delta.tables import DeltaTable

# check if Silver delta table exists or not

if spark.catalog.tableExists("workspace.silver.sales_enriched"):
    silver_table = DeltaTable.forName(spark, "workspace.silver.sales_enriched")
    (
        silver_table.alias("Target")
        .merge(
            df_silver.alias("Source"),
            "Target.order_id = Source.order_id"
        )
        .whenMatchedUpdateAll()
        .whenNotMatchedInsertAll()
        .execute()
    )
else:
    df_silver.write.format("delta") \
        .partitionBy("Order_year", "Order_month") \
        .saveAsTable("workspace.silver.sales_enriched")

print("Silver merge complete:", spark.table("workspace.silver.sales_enriched").count(), "rows")
#